# D05 – Streamlit Prototype
## The "Value-for-Money" Transfer Scout
*Group 7: Megann Kouandjeu, Salma Kamoun, Axel Chartier*

---

### Overview

This deliverable documents the mock-up Streamlit application (`app.py`) and the plan for replacing all mock/hardcoded elements with real model outputs.

The app is structured around the three features defined in D01's proposal:

| Feature | D01 Description | App Page |
|---|---|---|
| Budget Clone Finder | *"A 'Scouting Tool' where you select a famous player and the app suggests 3 budget versions with similar statistical profiles"* | 🔍 Budget Clone Finder |
| Scout Search | *"A 'Scout Search' where you input a budget and position and the app returns the top 5 undervalued gems"* | 💼 Scout Search |
| Player Risk Profile | *"A 'Player Risk Profile' showing scoring potential and physical fragility"* | Embedded in both tools |

**How to run the mock-up locally:**
```bash
pip install streamlit plotly pandas numpy
streamlit run app.py
```

---
## 1. App Pages — Screenshots Description

The app has four pages, accessible from the sidebar navigation.

---

### Page 1 — 🏠 Overview

**Purpose:** Communicate the business objective to a non-technical audience (a sporting director, not an ML engineer).

**What it shows:**
- A two-column layout: left side has the problem statement and tool descriptions; right side has 7 KPI cards (dataset size, model metrics, median value).
- A grouped bar chart titled *"How the Valuation Gap Works"* showing 5 mock players — the green bar (predicted value) extending beyond the red bar (asking price) is the buy signal.
- The chart immediately communicates the core concept: **gap = opportunity**.

**Design choices:**
- Dark theme (`#0f1117` background) — intentional for a professional scouting tool look.
- The mock RMSLE/R² numbers (0.81 / 0.74) are from D03's XGBoost baseline, not invented.

---

### Page 2 — 🔍 Budget Clone Finder

**Purpose:** The primary scouting use case. A club can't sign De Bruyne — who can they sign instead?

**What it shows:**
- A `selectbox` with 4 preset superstars (De Bruyne, Haaland, Vinícius Jr, Saka).
- The selected player's KPI bar: price, G+A per 90, minutes per game, injury risk.
- Three expandable **player cards** (one per budget clone), each containing:
  - Left panel: player details table with valuation gap pill and savings tag.
  - Centre panel: a **radar chart** (Plotly) comparing the superstar vs the clone on 5 dimensions: G+A/90, Minutes/Game, Age Score, Fitness (1 − injury_risk), Height.
  - Right panel: a **gauge chart** showing the clone's injury risk on a 0–100% dial with green/amber/red zones.

**Design choices:**
- The first clone is pre-expanded; others are collapsed — reduces visual overwhelm.
- Radar chart uses the same 5 features as the D02 §13 hypothesis features, making the visualisation directly linked to the model.
- Tags (`💰 BUY`, `🌟 Prime Age`, `⚠ Injury Risk`) encode the key business signals at a glance.

---

### Page 3 — 💼 Scout Search

**Purpose:** Budget-constrained search. Input budget + position → top 5 undervalued players.

**What it shows:**
- Three input widgets: position selectbox, budget slider (€2M–€50M), minimum minutes/game slider, maximum injury risk slider.
- A **horizontal grouped bar chart** (price in red, predicted value in green) — the green bar extending past the red visually encodes the valuation gap for all 5 results at once.
- Five **player rows** with medal emojis (🥇🥈🥉), player card, and four `st.metric` widgets: Asking Price, Predicted Value (+gap delta), G+A per 90, Minutes/Game, Potential ROI %.

**Design choices:**
- Results sorted by descending valuation gap — the biggest opportunities appear first.
- The ROI % metric (`(predicted − price) / price × 100`) is the D01 "Business Metric" made concrete.

---

### Page 4 — 📊 Model Story

**Purpose:** Provide transparency to the user about how the model was built and what it learned.

**What it shows:**
- Three **model comparison cards** (Ridge / Random Forest / XGBoost) with RMSLE, MAE, R² from D03, and a green border around XGBoost.
- A **horizontal bar chart** of XGBoost feature importance (gain) — using the mock values from D03's feature importance hypothesis in D02 §13.
- A vertical **pipeline summary** (D01 → D02 → D03 → D04 → D05) showing the full project narrative.
- A four-point **"Why XGBoost?"** summary with specific numbers.

**Design choices:**
- Feature importance bars are colour-coded by tier (green = top 33%, orange = mid, red = low).
- The pipeline summary makes the deliverables visible to the grader within the app itself.

---
## 2. Replacement Plan — Mock → Real

This section documents every hardcoded element in `app.py` and the exact code change needed to replace it with real model outputs.

---

### 2.1 Complete Replacement Map

| Mock Element | Location in `app.py` | Replacement Source | Replacement Code |
|---|---|---|---|
| `MOCK_SUPERSTARS` dict | Top of file | `df_engineered_outfield.csv` filtered to top market values | `df[df['market_value_eur'] > 50e6].sort_values('market_value_eur', ascending=False).head(20)` |
| `MOCK_BUDGET_CLONES` dict | Top of file | KNN similarity on XGBoost SHAP values | See §2.2 |
| `MOCK_SCOUT_RESULTS` dict | Top of file | Model predictions filtered by budget + position | See §2.3 |
| `FEATURE_IMPORTANCE` dict | Top of file | `xgb_model.feature_importances_` | `dict(zip(FEATURES_TREE, xgb_model.feature_importances_))` |
| KPI dataset stats (22,841 / 20,108) | Page 1 | `len(df_raw)` / `len(df_eng)` | `st.metric('Total Players', f'{len(df_raw):,}')` |
| KPI model metrics (R²=0.74, RMSLE=0.81) | Page 1 | Computed in D03, saved to `metrics.json` | `json.load(open('models/metrics.json'))['xgb']['rmsle']` |
| Valuation gap chart (Page 1) | Page 1 | Top 5 gaps from full predictions df | `df_preds.nlargest(5, 'valuation_gap')[['player_name','market_price','predicted_value']]` |
| Radar chart values | `radar_chart()` function | Actual per-90 stats from `df_engineered_outfield.csv` | Direct column lookup on `player_id` |
| Injury risk gauge | `risk_gauge()` function | `injury_risk_score` column from D02 | `df_eng.loc[df_eng.player_id == pid, 'injury_risk_score'].values[0]` |

---

### 2.2 Budget Clone Finder — Real Implementation

The mock-up uses hardcoded clone lists. The real implementation uses **cosine similarity on SHAP feature contribution vectors**:

```python
# STEP 1: Load model and data (do this once at app startup with @st.cache_resource)
import joblib, shap
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

@st.cache_resource
def load_model_and_data():
    xgb_model = joblib.load(r'..\models\xgb_model.pkl')
    df_eng    = pd.read_csv(r'..\datasets\df_engineered_outfield.csv')
    
    FEATURES_TREE = [
        'age', 'age_squared', 'prime_age_flag', 'height',
        'goals_per_90', 'assists_per_90', 'goal_contributions_per_90',
        'minutes_per_game_proxy', 'injury_risk_score', 'is_injury_prone',
        'is_left_footed', 'is_ambidextrous',
        'pos_ATT', 'pos_DEF', 'pos_MID', 'pos_WNG'
    ]
    FEATURES_TREE = [c for c in FEATURES_TREE if c in df_eng.columns]
    
    X = df_eng[FEATURES_TREE]
    
    # Compute SHAP values for all players
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X)              # shape: (n_players, n_features)
    
    # Model predictions → back-transform to euros
    df_eng['predicted_value'] = np.expm1(xgb_model.predict(X))
    df_eng['valuation_gap']   = df_eng['predicted_value'] - df_eng['market_value_eur']
    
    return xgb_model, df_eng, shap_values, FEATURES_TREE


# STEP 2: Find budget clones for a target player
def find_budget_clones(target_player_id, max_price_eur, df_eng, shap_values, n=3):
    """
    Finds the n most statistically similar players to the target
    who cost less than max_price_eur and are undervalued.
    """
    target_idx   = df_eng.index[df_eng['player_id'] == target_player_id][0]
    target_shap  = shap_values[target_idx].reshape(1, -1)
    
    # Cosine similarity of SHAP vectors → same features driving value
    similarities = cosine_similarity(target_shap, shap_values)[0]
    
    # Filter: cheaper than budget, same position group, undervalued (gap > 0)
    mask = (
        (df_eng['market_value_eur'] <= max_price_eur) &
        (df_eng['player_id'] != target_player_id) &
        (df_eng['valuation_gap'] > 0) &
        (df_eng['position_group'] == df_eng.loc[target_idx, 'position_group'])
    )
    
    candidate_idx   = df_eng.index[mask]
    candidate_sims  = similarities[candidate_idx]
    top_n_idx       = candidate_idx[np.argsort(candidate_sims)[::-1][:n]]
    
    return df_eng.loc[top_n_idx].copy()
```

---

### 2.3 Scout Search — Real Implementation

The mock-up returns hardcoded player lists. The real implementation filters and sorts the predictions dataframe:

```python
def scout_search(df_eng, position_group, budget_eur, min_mpg=55, max_injury_risk=0.4, top_n=5):
    """
    Returns the top_n most undervalued players matching the budget constraints.
    Requires df_eng to have 'predicted_value' and 'valuation_gap' columns
    computed from the trained model (see load_model_and_data above).
    """
    filtered = df_eng[
        (df_eng['position_group']    == position_group) &
        (df_eng['market_value_eur']  <= budget_eur) &
        (df_eng['minutes_per_game_proxy'] >= min_mpg) &
        (df_eng['injury_risk_score'] <= max_injury_risk) &
        (df_eng['valuation_gap']     > 0)                  # only undervalued
    ].copy()
    
    filtered['roi_pct'] = (filtered['valuation_gap'] / filtered['market_value_eur']) * 100
    
    return filtered.nlargest(top_n, 'valuation_gap')
```

---

### 2.4 Model and Data Loading — Startup Code

Replace the mock data block at the top of `app.py` with this:

```python
# ── Load real assets (replaces all MOCK_* dicts) ───────────────────────────
import joblib, json, shap
import numpy as np

@st.cache_resource
def load_assets():
    xgb_model = joblib.load(r'..\models\xgb_model.pkl')         # trained in D03
    df_raw    = pd.read_csv(r'..\datasets\master_training_data.csv')  # D01
    df_eng    = pd.read_csv(r'..\datasets\df_engineered_outfield.csv') # D02
    metrics   = json.load(open(r'..\models\metrics.json'))        # D03 eval results
    return xgb_model, df_raw, df_eng, metrics

@st.cache_data
def compute_predictions(_model, _df_eng):
    """Cache predictions so they don't recompute on every widget interaction."""
    FEATURES_TREE = [c for c in [
        'age', 'age_squared', 'prime_age_flag', 'height',
        'goals_per_90', 'assists_per_90', 'goal_contributions_per_90',
        'minutes_per_game_proxy', 'injury_risk_score', 'is_injury_prone',
        'is_left_footed', 'is_ambidextrous',
        'pos_ATT', 'pos_DEF', 'pos_MID', 'pos_WNG'
    ] if c in _df_eng.columns]
    X = _df_eng[FEATURES_TREE]
    _df_eng = _df_eng.copy()
    _df_eng['predicted_value'] = np.expm1(_model.predict(X))
    _df_eng['valuation_gap']   = _df_eng['predicted_value'] - _df_eng['market_value_eur']
    return _df_eng

xgb_model, df_raw, df_eng, metrics = load_assets()
df_eng = compute_predictions(xgb_model, df_eng)
```

> **Note on `@st.cache_resource` vs `@st.cache_data`:**
> - `@st.cache_resource` is used for the model (a non-serialisable object like a scikit-learn/XGBoost estimator). It is loaded once and shared across all user sessions.
> - `@st.cache_data` is used for the predictions dataframe — it is serialised and cached per unique function input.

---
## 3. File to Save from D03 — `metrics.json` and `xgb_model.pkl`

The replacement plan requires two files to be saved at the end of D03 (or the final modelling notebook):

```python
# Add this to the end of D03 after training and evaluating the models

import joblib, json, os
os.makedirs(r'..\models', exist_ok=True)

# Save the trained XGBoost model
joblib.dump(xgb_model, r'..\models\xgb_model.pkl')

# Save evaluation metrics for the Streamlit KPI cards
metrics = {
    'ridge' : {'rmsle': r_rmsle, 'mae_eur': r_mae, 'r2': r_r2},
    'rf'    : {'rmsle': f_rmsle, 'mae_eur': f_mae, 'r2': f_r2},
    'xgb'   : {'rmsle': x_rmsle, 'mae_eur': x_mae, 'r2': x_r2},
}
with open(r'..\models\metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Model and metrics saved to ..\\models\\')
```

---
## 4. Summary — Mock-up vs Final App

| Component | Mock-up (D05) | Final App |
|---|---|---|
| Player data | Hardcoded dicts (`MOCK_SUPERSTARS`, etc.) | Live query on `df_engineered_outfield.csv` |
| Predicted values | Manually estimated numbers | `np.expm1(xgb_model.predict(X))` |
| Valuation gap | Hardcoded | `predicted_value − market_value_eur` |
| Budget clone search | Pre-selected hardcoded alternatives | KNN on SHAP cosine similarity |
| Scout search | Hardcoded filtered list | `df_eng.nlargest(5, 'valuation_gap')` with filters |
| Feature importance | Hardcoded dict | `dict(zip(FEATURES_TREE, xgb_model.feature_importances_))` |
| Model metrics (KPIs) | From D03 results (typed in) | `json.load(open('models/metrics.json'))` |
| Radar chart values | Computed from hardcoded player dicts | Computed from `df_eng` row lookup by `player_id` |
| Injury risk gauge | Hardcoded `injury_risk` value | `df_eng.loc[mask, 'injury_risk_score']` from D02 |
| UI / layout | ✅ Final — no changes needed | ✅ No changes — CSS, structure, charts stay the same |
| Pages / navigation | ✅ Final — no changes needed | ✅ No changes |

**The UI layer is complete and final.** Only the data layer changes — every mock dict is replaced by a direct lookup or model call. The Streamlit widget logic, CSS, chart types, and page structure require no modification.